# Random Forest Classifier

Part 1: Predicting if a Storm Would Occur using Random Forest Classifier

Input: Year, Month, Day, CO2 emission (Tons), and Monthly global temperatures 
Objective: min Z1 classification error, when predicting tropical storms
Constraints: CO2 emissions, Monthly global temperatures, Year, Month, and Day >= 0 and If Storm Intensity Label > 1 && Storm Intensity Label >=7 then “storm_occured” = 1 else 0

Expected output - binary value 1 for tropical storm occurred else 0

In [35]:
#all imports
import pandas as pd
import numpy as np
from itertools import product
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.utils import shuffle
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
import time
from sklearn.metrics import accuracy_score, precision_score, mean_squared_error, mean_absolute_error, r2_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.pipeline import Pipeline as ImbPipeline

In [36]:
#Reading from the dataset
data = pd.read_csv("completed_dataset_for_IS_project_25.csv")

#testing if the reading from the dataset was successful
print(data.head(5))

   Year  MONTH  DAY   LAT  LONG  WIND_KTS  PRESSURE CAT  Shape_Leng Country  \
0  1880      8   11  23.0 -91.9        70         0  H1    0.806226  Mexico   
1  1880      8   11  23.4 -92.6        80         0  H1    0.761577  Mexico   
2  1880      8   11  23.7 -93.3        80         0  H1    0.583095  Mexico   
3  1880      8   12  24.0 -93.8        90         0  H2    0.670820  Mexico   
4  1880      9    6  23.9 -88.6        40         0  TS    0.360555  Mexico   

   ...   Jun   Jul   Aug   Sep   Oct   Nov   Dec  Storm Intensity  \
0  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         56.43582   
1  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.92616   
2  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         46.64760   
3  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         60.37380   
4  ... -0.21 -0.18 -0.11 -0.15 -0.24 -0.22 -0.18         14.42220   

   Storm Intensity Label  Wind Speed Squared  
0                      3                4900  
1               

Using the data in the dataset where the Storm Intensity Label is greater than 1 and less than 7 to represent tropical storm records the the lables less than and equal to 1 to represent the data for no storm occured 

In [37]:
data["storm_occured"] = data["Storm Intensity Label"].apply(lambda val: 1 if 1 < val <= 7 else 0)

Checking if the no storm data was added into the dataset

In [38]:
no_storm = data[data["storm_occured"]== 0]
yes_storm = data[data["storm_occured"]== 1]

print(no_storm.shape, yes_storm.shape)

(14017, 27) (38639, 27)


The data is unbalanced leaning to the yes storm data which would cause our model to have a bias to yes storm, will have to balance the data for a better prediction 

In [39]:
yes_storm = yes_storm.head(no_storm.shape[0])
print(no_storm.shape, yes_storm.shape)

(14017, 27) (14017, 27)


Joining and shuffling the remaining data

In [40]:
data = pd.concat([no_storm, yes_storm])
data = shuffle(data)
data.head(5)

,Year,MONTH,DAY,LAT,LONG,WIND_KTS,PRESSURE,CAT,Shape_Leng,Country,...,Jul,Aug,Sep,Oct,Nov,Dec,Storm Intensity,Storm Intensity Label,Wind Speed Squared,storm_occured
39008,1991,7,1,27.3,-82.7,20,1012,L,0.700000,United States,...,0.47,0.39,0.44,0.29,0.30,0.32,14.000000,0,400,0
36265,1988,10,30,12.7,-108.0,10,0,TD,1.019804,Mexico,...,0.33,0.39,0.37,0.38,0.12,0.29,10.198040,1,100,0
25651,1971,9,9,27.0,-67.6,25,1008,TD,0.360555,Bermuda,...,-0.08,-0.01,-0.06,-0.04,-0.07,-0.08,9.013875,1,625,0
686,1883,10,26,44.0,-46.0,60,0,E,5.852350,Canada,...,-0.08,-0.14,-0.23,-0.12,-0.24,-0.11,351.141000,-1,3600,0
7372,1911,8,28,32.1,-80.3,85,972,H2,0.707107,United States,...,-0.41,-0.44,-0.41,-0.27,-0.21,-0.22,60.104095,4,7225,1


Spliting the data

In [41]:
features = ["Year", "MONTH", "DAY", "CO2 emission (Tons)", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
target = "storm_occured"

X= data[features]
y = data[target]

#Spliting the merged dataset into training (70%), validation (15%), and testing (15%) sets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=5, stratify=y) #for balance split
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=5, stratify=y_temp)

#testing splits
print(f"Total size: {len(X)}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Total size: 28034
Train size: 19623
Validation size: 4205
Test size: 4206


Training the Model

In [42]:
#dropping rows with missing values
X_train, y_train = X_train.dropna(), y_train.loc[X_train.dropna().index]
X_val, y_val = X_val.dropna(), y_val.loc[X_val.dropna().index]
X_test, y_test = X_test.dropna(), y_test.loc[X_test.dropna().index]

print(y_train.value_counts())

model = RandomForestClassifier(n_estimators=100, random_state=5, max_features= 5)
model.fit(X_train, y_train)

storm_occured
0    9812
1    9811
Name: count, dtype: int64


RandomForestClassifier(max_features=5, random_state=5)

Evaluation using the test and validation data

Classification → Accuracy, Precision
Efficiency → Latency


In [43]:
#Test
start_time_test = time.time()
y_test_prediction = model.predict(X_test)
latency_test = time.time() - start_time_test

accuracy_test = accuracy_score(y_test, y_test_prediction)
precision_test = precision_score(y_test, y_test_prediction)

print("Evaluation Test")
print(f"Accuracy: {accuracy_test}")
print(f"Precision: {precision_test}")
print(f"Latency: {latency_test}")

Evaluation Test
Accuracy: 0.9612458392772231
Precision: 0.9503249767873724
Latency: 0.08991575241088867


In [44]:
#Validation
start_time_val = time.time()
y_val_prediction = model.predict(X_val)
latency_val = time.time() - start_time_val

accuracy_val = accuracy_score(y_val, y_val_prediction)
precision_val = precision_score(y_val, y_val_prediction)

print("Evaluation Validation")
print(f"Accuracy: {accuracy_val}")
print(f"Precision: {precision_val}")
print(f"Latency: {latency_val}")

Evaluation Validation
Accuracy: 0.9619500594530321
Precision: 0.9558892538714219
Latency: 0.0939488410949707


For the user to add in their variables to predict if a storm would occur or not and if a storm does occur it would lead the user into the other model to predict the intensity

In [45]:
def predict_storm(model, year, month, day, co2, temps_dict):
    input = {
        "Year": [year],
        "MONTH": [month],
        "DAY": [day],
        "CO2 emission (Tons)": [co2],
    }

    months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov","Dec"]
    
    for month_temp in months:
        input[month_temp] = [temps_dict.get(month_temp, 0)]

    input_data = pd.DataFrame(input)
    prediction = model.predict(input_data)[0]

    return prediction #0 or 1 for the model in part 2

In [46]:
#example for predicting a possible storm
possible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}
will_storm_occur = predict_storm(model=model, year=1880, month=8, day=9, co2= 0.0, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")

A storm was predicted


In [ ]:
#example for predicting a no storm
possible_temps= {"Jan": -1.5, "Feb": -1.3, "Mar": -1.1, "Apr": -1.2, "May": -1.0, "Jun": -1.4, "Jul": -1.3, "Aug": -1.5, "Sep": -1.2, "Oct": -1.3, "Nov": -1.4, "Dec": -1.6}
will_storm_occur = predict_storm(model=model, year=1890, month=1, day=5, co2= 15000000.0, temps_dict = possible_temps)
print("A storm was predicted" if will_storm_occur == 1 else "No storm was predicted")

No storm was predicted


In [48]:
print("Classification Report for Model 1 - Gradient Boosting Classifier")

print("Test Report")
print(classification_report(y_test, y_test_prediction))

print("Validation Report")
print(classification_report(y_val, y_val_prediction))

Classification Report for Model 1 - Gradient Boosting Classifier
Test Report
              precision    recall  f1-score   support

           0       0.97      0.95      0.96      2103
           1       0.95      0.97      0.96      2103

    accuracy                           0.96      4206
   macro avg       0.96      0.96      0.96      4206
weighted avg       0.96      0.96      0.96      4206

Validation Report
              precision    recall  f1-score   support

           0       0.97      0.96      0.96      2102
           1       0.96      0.97      0.96      2103

    accuracy                           0.96      4205
   macro avg       0.96      0.96      0.96      4205
weighted avg       0.96      0.96      0.96      4205



# Random Forest Regressor

Part 2: Predicting the intensity of the storm using Random Forest Regressor

Objective: 
    min Z2 error in predicting the intensity of the tropical storm
Constraints:
	    pressure>= 0 
        stormed_occured =1
	    Storm Intensity Label > 1 && Storm Intensity Label <=7

Ensures the intensity model trains only where tropical storms occur by filtering to 'Storm Intensity Label' > 1

In [15]:
train_storm_indices = y_train[y_train == 1].index
train_storm_data = data.loc[train_storm_indices]
train_storm_data = train_storm_data[(train_storm_data['Storm Intensity Label'] > 1) & (train_storm_data['Storm Intensity Label'] <= 7)]
# train_storm_data = train_storm_data[train_storm_data['Storm Intensity Label'] > 1] #adjust this to match this "Storm Intensity Label > 1 && Storm Intensity Label <=7"



In [16]:
print("\nColumns in X_test_occ before dropping:")
print(X_test.columns)


Columns in X_test_occ before dropping:
Index(['Year', 'MONTH', 'DAY', 'CO2 emission (Tons)', 'Jan', 'Feb', 'Mar',
       'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
      dtype='object')


In [17]:
if not train_storm_data.empty:
    X_train_int = train_storm_data.drop(columns=['storm_occured', 'Country', 'CAT', 'Storm Intensity', 'Storm Intensity Label','WIND_KTS', 'Wind Speed Squared', 'PRESSURE', 'LAT', 'LONG', 'Shape_Leng'])
    y_train_int = train_storm_data['Storm Intensity Label']
    print(X_train_int.columns)

    print("Original intensity distribution:")
    print(y_train_int.value_counts().sort_index())

    # Intensity model pipeline with SMOTE to balance the imbalance in the intensity classes, while using the random forest classifier for the non-linear relationships and the standard scaler to normalize the numerical features 
    intensity_pipeline = ImbPipeline([
        ('scaler', StandardScaler()),
        # ('smote', SMOTE(random_state=42)),
        ('classifier', 
            RandomForestRegressor(random_state=42)
        )
    ])

    #Hyperparameter tuning to ajust tree depth and ensemble sizes for Random Forrest classifier model
    param_grid_int = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10]
    }

    # f1_weighted_scorer = make_scorer(f1_score, average='weighted')

    grid_search_int = GridSearchCV(
        intensity_pipeline,
        param_grid_int,
        cv=5,
        # scoring=make_scorer(f1_score, average='weighted'),
        # scoring=f1_weighted_scorer,
        scoring='neg_mean_squared_error',
        n_jobs=2,
        verbose=2,
        error_score='raise'
    )
    grid_search_int.fit(X_train_int, y_train_int)
    best_intensity_model = grid_search_int.best_estimator_
    
    #Evaluates on storms predicted by the model earlier while further filtering out the storm intensities
    predicted_storm_indices = y_test[y_test == 1].index
    if len(predicted_storm_indices) > 0:
        # Get true intensity labels for these storms
        y_test_intensity = data.loc[predicted_storm_indices, 'Storm Intensity Label']
        real_storms = y_test_intensity[y_test_intensity > 1]
        
        if len(real_storms) > 0:
                       
            # X_test_storms = X_test_occ.loc[real_storms.index]
            X_test_storms = X_test.loc[real_storms.index, X_train_int.columns]

            # Final Evaluation to show how model performed for classifying the different levels of a tropical storm
            y_pred_intensity = best_intensity_model.predict(X_test_storms)
           
            
            print("\nTrue intensity distribution in test set:")
            print(real_storms.value_counts().sort_index())
            
            # print("\nIntensity Prediction Report:")
            # print(classification_report(real_storms, y_pred_intensity))
            mse = mean_squared_error(real_storms, y_pred_intensity)
            mae = mean_absolute_error(real_storms, y_pred_intensity)
            r2 = r2_score(real_storms, y_pred_intensity)

            print("\nIntensity Prediction Metrics:")
            print(f"Mean Squared Error (MSE): {mse:.2f}")
            print(f"Mean Absolute Error (MAE): {mae:.2f}")
            print(f"R-squared (R2): {r2:.2f}")
        else:
            print("\nNo actual storms with intensity >1 in test set predictions")
    else:
        print("\nNo storms predicted in test set")
    # Evaluating on validation set (separate from test set)
    predicted_val_storm_indices = y_val[y_val == 1].index
    if len(predicted_val_storm_indices) > 0:
        y_val_intensity = data.loc[predicted_val_storm_indices, 'Storm Intensity Label']
        real_val_storms = y_val_intensity[y_val_intensity > 1]
        
        if len(real_val_storms) > 0:
            X_val_storms = X_val.loc[real_val_storms.index, X_train_int.columns]
            y_pred_intensity_val = best_intensity_model.predict(X_val_storms)
            
            print("\nTrue intensity distribution in validation set:")
            print(real_val_storms.value_counts().sort_index())
            
            mse_val = mean_squared_error(real_val_storms, y_pred_intensity_val)
            mae_val = mean_absolute_error(real_val_storms, y_pred_intensity_val)
            r2_val = r2_score(real_val_storms, y_pred_intensity_val)

            print("\nIntensity Prediction Metrics for validation:")
            print(f"Mean Squared Error (MSE): {mse_val:.2f}")
            print(f"Mean Absolute Error (MAE): {mae_val:.2f}")
            print(f"R-squared (R2): {r2_val:.2f}")
        else:
            print("\nNo actual storms with intensity >1 in validation set predictions")
    else:
        print("\nNo storms predicted in validation set")
else:
    print("No training data with intensity >1 available")


Index(['Year', 'MONTH', 'DAY', 'CO2 emission (Tons)', 'Jan', 'Feb', 'Mar',
       'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
      dtype='object')
Original intensity distribution:
Storm Intensity Label
2    5536
3    2189
4    1256
5     620
6     191
7      19
Name: count, dtype: int64
Fitting 5 folds for each of 4 candidates, totalling 20 fits

True intensity distribution in test set:
Storm Intensity Label
2    1182
3     444
4     274
5     146
6      52
7       5
Name: count, dtype: int64

Intensity Prediction Metrics:
Mean Squared Error (MSE): 0.26
Mean Absolute Error (MAE): 0.28
R-squared (R2): 0.78

True intensity distribution in validation set:
Storm Intensity Label
2    1204
3     439
4     276
5     133
6      45
7       6
Name: count, dtype: int64

Intensity Prediction Metrics for validation:
Mean Squared Error (MSE): 0.32
Mean Absolute Error (MAE): 0.30
R-squared (R2): 0.72


In [18]:
def predict_intensity(model, co2, year, month, day, temps_dict):
    #cat_int = le.transform([cat])[0]
    input = pd.DataFrame([{
    #"CAT_int": cat_int,
    "Year": year,
    "MONTH":month,
    "DAY": day,
    "CO2 emission (Tons)": co2,
    "Jan": temps_dict["Jan"],
    "Feb": temps_dict["Feb"],
    "Mar": temps_dict["Mar"],
    "Apr": temps_dict["Apr"],
    "May": temps_dict["May"],
    "Jun": temps_dict["Jun"],
    "Jul": temps_dict["Jul"],
    "Aug": temps_dict["Aug"],
    "Sep": temps_dict["Sep"],
    "Oct": temps_dict["Oct"],
    "Nov": temps_dict["Nov"],
    "Dec": temps_dict["Dec"]
    #"WIND_KTS": wind_kts
    }])

    return model.predict(input)[0]

In [19]:
possible_temps= {"Jan": -0.19, "Feb": -0.25, "Mar": -0.09, "Apr": -0.17, "May": -0.10, "Jun": -0.21, "Jul": -0.18, "Aug": -0.11, "Sep": -0.15, "Oct": -0.24, "Nov": -0.22, "Dec": -0.18}
will_storm_occur = predict_storm(model=model, year=1880, month=8, day=14, co2= 2667491000.0, temps_dict = possible_temps)

if will_storm_occur == 1:
    print("A storm was predicted, now predicting it's intensity...")
    #intensity = predict_intensity(model = model_2, wind_kts=70, cat= "H2", co2=412.5, year=2024, month=8, day=14)
    #intensity = predict_intensity(model = model_2, wind_kts=70, co2=412.5, year=2024, month=8, day=14)
    intensity = predict_intensity(model = best_intensity_model, co2=412.5, year=2024, month=8, day=14, temps_dict = possible_temps)

    #Before rounding
    #print(f"Predicted storm intensity: {intensity}")
    #print("Based on MyNASEData Hurricane Dynamics")
    
    #Some times I get the intensity as a decimal, using this to round to the nearest whole number
    intensity = round(intensity)
    
    if intensity == -2:
        value = "Warning flag"
    elif intensity == -1:
        value = "Extra-tropical (non-standard storm)"
    elif intensity == 0:
        value = "Low-pressure system"
    elif intensity == 1:
        value = "Tropical Depression"
    elif intensity == 2:
        value = "Tropical Storm"
    elif intensity == 3:
        value = "Hurricane Category 1"
    elif intensity == 4:
        value = "Hurricane Category 2"
    elif intensity == 5:
        value = "Hurricane Category 3"
    elif intensity == 6:
        value = "Hurricane Category 4"
    elif intensity == 7:
        value = "Hurricane Category 5"
    else:
        value = "Unknown"

    print(f"Predicted storm intensity: {value} with an intensity of {intensity}")
else:
    print(f"No storm was predicted, skipping the intensity prediction")

A storm was predicted, now predicting it's intensity...
Predicted storm intensity: Hurricane Category 1 with an intensity of 3
